# Comparing all wind-offshore approaches, plus SMARD

Same comparison as onshore's `12_compare_all_approaches`, adapted to three
approaches (there's no offshore equivalent of the `delu-headline-forecast`
sibling project's baseline to reuse as a fourth):

1. **PECD's own PEOF zone capacity factor x MaStR zone capacity** --
   trusts PECD's official numbers wholesale, just reweights them by
   MaStR's installed capacity.
2. **This project's own naive physics baseline** (`pipeline/20`) --
   computed from scratch with the simplest possible physics: one generic
   turbine power curve for the entire fleet, raw ERA5, no bias
   correction. This is the "without PECD data" replication.
3. **This project's PECD replication** (`pipeline/30`-`32`) -- computed
   from scratch with PECD's actual documented methodology as closely as
   achievable: GWA2 bias-adjusted wind speed where available (else raw
   ERA5 -- see `31`'s docstring), alpha-based hub-height extrapolation,
   per-plant power curves. No storm-shutdown derate (see `31`/`32`'s
   docstrings for why).

Plus **SMARD's actual DE-LU wind-offshore generation** as the real-world
ground truth none of the three are derived from.

All series use a fleet-vintage-correct weighting: approaches 1 and 2, and
the SMARD-implied CF, use MaStR's *time-varying* monthly PEOF capacity as
the weighting denominator (`pecdr.capacity_weighting`); approach 3 runs
each year's weather through **that year's own** start-of-year fleet
(`pipeline/30`, `pipeline/32`) rather than today's fleet applied
retroactively.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pecdr.capacity_weighting import capacity_weighted_cf, total_hourly_capacity
from pecdr.domain import FULL_YEAR_BACKTEST_YEARS
from pecdr.paths import ProjPaths

paths = ProjPaths()
paths.ensure_directories()

PERIOD_LABEL = f"{FULL_YEAR_BACKTEST_YEARS[0]}-{FULL_YEAR_BACKTEST_YEARS[-1]}"

## Load all series, all as capacity factor, restricted to the common backtest period

One data-quality wrinkle in approach 1: 3 of PECD's 6 PEOF zones
(`DE013_OFF`/`DE014_OFF`/`DE015_OFF`) are **100% `NaN`** in PECD's own
product across the entire 2015-2025 file -- PECD apparently never
modeled them. `DE013_OFF` still carries real MaStR capacity (up to
~300 MW). Passing all 6 zones into `capacity_weighted_cf` unfiltered
would silently dilute its capacity-weighted denominator with this
dead-weight capacity -- `DE013_OFF`'s real MW counts toward "total
capacity" but contributes nothing to the numerator, since pandas'
default `skipna` sum just drops its `NaN` CF -- systematically pulling
approach 1's implied CF down for no real-world reason. Restricting to
the zones PECD actually has data for avoids that; found and confirmed
while building notebook 35's capacity-fitting exercise.

In [2]:
mastr_peof_month = pd.read_parquet(paths.mastr_capacity_by_peof_month_file)

# Approach 1: PECD's own PEOF zone CF x time-varying MaStR PEOF capacity,
# restricted to the zones PECD actually models (see note above).
pecd_peof_cf = pd.read_parquet(paths.pecd_wind_offshore_capacity_factors_file)
pecd_modeled_zones = pecd_peof_cf.columns[pecd_peof_cf.notna().any()].tolist()
approach1 = capacity_weighted_cf(pecd_peof_cf[pecd_modeled_zones], mastr_peof_month)

# Approach 2: this project's own raw-ERA5 + generic-curve PEOF zone CF, same weights
own_naive_cf = pd.read_parquet(paths.own_wind_offshore_capacity_factors_file)
own_naive_cf.index.name = "timestamp"
approach2 = capacity_weighted_cf(own_naive_cf, mastr_peof_month)

# Approach 3: this project's replication (already national, each year's own fleet)
approach3 = pd.read_parquet(paths.wind_offshore_full_year_cf_file)["national"]

# SMARD actual generation (MW) -> implied CF via time-varying total capacity
target_panel = pd.read_parquet(paths.target_panel_file)
total_capacity_hourly = total_hourly_capacity(mastr_peof_month, target_panel.index)
smard_implied_cf = target_panel["wind_offshore"] / total_capacity_hourly

comparison = pd.DataFrame({
    "1_pecd_official_x_mastr": approach1,
    "2_naive_physics_no_pecd": approach2,
    "3_pecd_replication": approach3,
    "4_smard_implied": smard_implied_cf,
})
comparison = comparison.loc[FULL_YEAR_BACKTEST_YEARS[0]:FULL_YEAR_BACKTEST_YEARS[-1]].dropna(how="all")
print(f"Hours with at least one series: {len(comparison):,}")
comparison.describe()

Hours with at least one series: 61,368


,1_pecd_official_x_mastr,2_naive_physics_no_pecd,3_pecd_replication,4_smard_implied
count,61368.000000,61368.000000,61368.000000,61368.000000
mean,0.424536,0.421876,0.459160,0.351297
std,0.291658,0.354737,0.330094,0.237328
min,0.000000,0.000000,0.000000,0.000000
25%,0.156340,0.084532,0.132425,0.134232
50%,0.378124,0.323454,0.431495,0.326057
75%,0.697192,0.783224,0.802703,0.557530
max,0.960379,1.000000,0.900296,0.918114


## Full-period time series, all four series

In [3]:
fig, ax = plt.subplots(figsize=(14, 5))
for col, label in [
    ("1_pecd_official_x_mastr", "1. PECD official x MaStR"),
    ("2_naive_physics_no_pecd", "2. Naive physics (no PECD data)"),
    ("3_pecd_replication", "3. This project's PECD replication"),
    ("4_smard_implied", "4. SMARD actual (implied CF)"),
]:
    comparison[col].plot(ax=ax, label=label, alpha=0.7, linewidth=0.3)
ax.set_ylabel("Germany-wide offshore wind capacity factor")
ax.set_title(f"All four approaches, {PERIOD_LABEL}")
ax.legend()
fig.tight_layout()
fig.savefig(paths.images_path / "34_all_approaches_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

/tmp/ipykernel_589074/857773035.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


```{figure} ../../output/images/34_all_approaches_timeseries.png
:name: fig-34-all-approaches-timeseries
All four wind-offshore capacity factor series, 2019-2025.
```

## Each modeled approach vs. SMARD's actual (implied) capacity factor

In [4]:
stats_rows = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, label in zip(axes, ["1_pecd_official_x_mastr", "2_naive_physics_no_pecd", "3_pecd_replication"], ["1. PECD official x MaStR", "2. Naive physics (no PECD)", "3. PECD replication"]):
    sub = comparison[[col, "4_smard_implied"]].dropna()
    diff = sub[col] - sub["4_smard_implied"]
    corr = sub[col].corr(sub["4_smard_implied"])
    stats_rows.append({"approach": label, "n_hours": len(sub), "mae": float(diff.abs().mean()), "bias": float(diff.mean()), "corr": float(corr)})

    ax.scatter(sub["4_smard_implied"], sub[col], s=2, alpha=0.05)
    lims = [0, max(sub.max())]
    ax.plot(lims, lims, "k--", alpha=0.5, label="1:1")
    ax.set_xlabel("SMARD-implied CF")
    ax.set_ylabel(f"{label} CF")
    ax.set_title(f"{label}\nMAE={diff.abs().mean():.3f}, corr={corr:.3f}")
fig.tight_layout()
fig.savefig(paths.images_path / "34_approaches_vs_smard.png", dpi=150, bbox_inches="tight")
plt.show()

stats = pd.DataFrame(stats_rows)
stats

/tmp/ipykernel_589074/3270078833.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,approach,n_hours,mae,bias,corr
0,1. PECD official x MaStR,61368,0.093568,0.073239,0.902358
1,2. Naive physics (no PECD),61368,0.133096,0.070579,0.882324
2,3. PECD replication,61368,0.128137,0.107863,0.916476


```{figure} ../../output/images/34_approaches_vs_smard.png
:name: fig-34-approaches-vs-smard
Each modeled approach vs. SMARD's actual (implied) capacity factor, 2019-2025.
```

## Does each approach's fit to SMARD hold up year to year?

In [5]:
yearly_rows = []
for col, label in [
    ("1_pecd_official_x_mastr", "1. PECD official x MaStR"),
    ("2_naive_physics_no_pecd", "2. Naive physics (no PECD)"),
    ("3_pecd_replication", "3. PECD replication"),
]:
    sub = comparison[[col, "4_smard_implied"]].dropna()
    diff = sub[col] - sub["4_smard_implied"]
    for year, group in diff.groupby(diff.index.year):
        approach_group = sub.loc[group.index, col]
        smard_group = sub.loc[group.index, "4_smard_implied"]
        yearly_rows.append({
            "approach": label,
            "year": year,
            "n_hours": len(group),
            "mae": float(group.abs().mean()),
            "bias": float(group.mean()),
            "corr": float(approach_group.corr(smard_group)),
        })

yearly_stats = pd.DataFrame(yearly_rows).pivot(index="year", columns="approach", values=["bias", "mae", "corr"])
yearly_stats

bias                             \
approach 1. PECD official x MaStR 2. Naive physics (no PECD)   
year                                                           
2019                     0.045825                   0.044836   
2020                     0.050169                   0.055044   
2021                     0.050029                   0.045771   
2022                     0.072200                   0.075133   
2023                     0.105035                   0.104062   
2024                     0.103449                   0.100780   
2025                     0.085945                   0.068388   

                                                  mae  \
approach 3. PECD replication 1. PECD official x MaStR   
year                                                    
2019                0.081400                 0.073098   
2020                0.089501                 0.076490   
2021                0.081183                 0.076345   
2022                0.108332                 0.093218   
2023                0.136537                 0.120284   
2024                0.141434                 0.117216   
2025                0.116615                 0.098306   

                                                         \
approach 2. Naive physics (no PECD) 3. PECD replication   
year                                                      
2019                       0.116673            0.105939   
2020                       0.120729            0.110864   
2021                       0.112600            0.103561   
2022                       0.136210            0.128765   
2023                       0.161673            0.155378   
2024                       0.157174            0.159178   
2025                       0.126578            0.133236   

                             corr                             \
approach 1. PECD official x MaStR 2. Naive physics (no PECD)   
year                                                           
2019                     0.936821                   0.924381   
2020                     0.942606                   0.925453   
2021                     0.933051                   0.911782   
2022                     0.928866                   0.909590   
2023                     0.856334                   0.828063   
2024                     0.852719                   0.828047   
2025                     0.895189                   0.872265   

                              
approach 3. PECD replication  
year                          
2019                0.947235  
2020                0.950987  
2021                0.942059  
2022                0.938597  
2023                0.880952  
2024                0.875533  
2025                0.913169